In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import os

def parse_champsim_output(file_path):
    """Parses the ChampSim output file to extract relevant metrics."""
    metrics = {}

    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found!")
        return metrics

    with open(file_path, 'r') as file:
        for line in file:
            if "cumulative IPC" in line:
                match = re.findall(r"\d+\.\d+", line)
                if match:
                    metrics['IPC'] = float(match[0])
            elif "AVERAGE MISS LATENCY" in line and "L2C" in line:
                match = re.findall(r"\d+\.\d+|\d+", line) 
                if match:
                    metrics['L2C Avg Miss Latency'] = float(match[-1]) 
            elif "L2C PREFETCH" in line and "ACCESS" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 4:
                    metrics['L2_PREFETCH Access'] = int(values[-4])  
                    metrics['L2_PREFETCH Hit'] = int(values[-3])    
                    metrics['L2_PREFETCH Miss'] = int(values[-2]) 
            elif "L2C PREFETCH" in line and "REQUESTED" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 5:
                    metrics['L2_PREFETCH Requested'] = int(values[-4])  
                    metrics['L2_PREFETCH Issued'] = int(values[-3])    
                    metrics['L2_PREFETCH Useful'] = int(values[-2]) 
                    metrics['L2_PREFETCH Useless'] = int(values[-1])    

    return metrics

In [4]:
prefetchers = ['ip-stride', 'next-line', 'no', 'va-ampm-lite', 'adapt-dist']
bfs_results = {}
dfs_results = {}
spmv_results = {}

for p in prefetchers:
    bfs_results[p] = parse_champsim_output(f'output/{p}/bfs_{p}.txt')
    dfs_results[p] = parse_champsim_output(f'output/{p}/dfs_{p}.txt')
    spmv_results[p] = parse_champsim_output(f'output/{p}/spmv_{p}.txt')

bfs_df = pd.DataFrame(bfs_results)
bfs_df['Benchmark'] = 'bfs'
dfs_df = pd.DataFrame(dfs_results)  
dfs_df['Benchmark'] = 'dfs'
spmv_df = pd.DataFrame(spmv_results)
spmv_df['Benchmark'] = 'spmv'

df = pd.concat([bfs_df, dfs_df, spmv_df])
df_long = df.reset_index().melt(id_vars=['index', 'Benchmark'], var_name='Prefetcher', value_name='Value')
df_long.rename(columns={'index': 'Metric'}, inplace=True)

In [5]:
df_long

,Metric,Benchmark,Prefetcher,Value
0,IPC,bfs,ip-stride,0.9198
1,L2_PREFETCH Access,bfs,ip-stride,3758.0000
2,L2_PREFETCH Hit,bfs,ip-stride,1156.0000
3,L2_PREFETCH Miss,bfs,ip-stride,2602.0000
4,L2_PREFETCH Requested,bfs,ip-stride,2774.0000
...,...,...,...,...
130,L2_PREFETCH Requested,spmv,adapt-dist,10714.0000
131,L2_PREFETCH Issued,spmv,adapt-dist,9260.0000
132,L2_PREFETCH Useful,spmv,adapt-dist,2776.0000
133,L2_PREFETCH Useless,spmv,adapt-dist,1697.0000


In [6]:
metrics = df_long['Metric'].unique()
metrics

array(['IPC', 'L2_PREFETCH Access', 'L2_PREFETCH Hit', 'L2_PREFETCH Miss',
       'L2_PREFETCH Requested', 'L2_PREFETCH Issued',
       'L2_PREFETCH Useful', 'L2_PREFETCH Useless',
       'L2C Avg Miss Latency'], dtype=object)

In [8]:
metrics = df_long['Metric'].unique()
for metric in metrics:
    if "PREFETCH" not in metric.upper():
        print(metric)

IPC
L2C Avg Miss Latency


In [10]:
import plotly.express as px

prefetch_colors={
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
    "no": "rgb(144, 238, 144)"
}

for metric in metrics:
    if "PREFETCH" not in metric.upper():
        df_metric = df_long[df_long['Metric'] == metric]
    
        fig = px.bar(
            df_metric, 
            x="Benchmark", 
            y="Value", 
            color="Prefetcher", 
            barmode="group", 
            title=f"{metric} Across Benchmarks",
            labels={"Value": metric, "Benchmark": "Benchmark", "Prefetcher": "Prefetcher"},
            color_discrete_map=prefetch_colors
        )

        if not df_metric.empty:
            fig.show()

In [12]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import os

def parse_champsim_output(file_path, benchmark, prefetcher):
    """Parses the ChampSim output file to extract relevant metrics."""
    metrics = {"Benchmark": benchmark, "Prefetcher": prefetcher}

    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found!")
        return metrics

    with open(file_path, 'r') as file:
        for line in file:
            if "cumulative IPC" in line:
                match = re.findall(r"\d+\.\d+", line)
                if match:
                    metrics['IPC'] = float(match[0])
            elif "AVERAGE MISS LATENCY" in line and "L2C" in line:
                match = re.findall(r"\d+\.\d+|\d+", line) 
                if match:
                    metrics['L2C Avg Miss Latency'] = float(match[-1]) 
            elif "L2C PREFETCH" in line and "ACCESS" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 4:
                    metrics['L2_PREFETCH Access'] = int(values[-4])  
                    metrics['L2_PREFETCH Hit'] = int(values[-3])    
                    metrics['L2_PREFETCH Miss'] = int(values[-2]) 
            elif "L2C PREFETCH" in line and "REQUESTED" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 5:
                    metrics['L2_PREFETCH Requested'] = int(values[-4])  
                    metrics['L2_PREFETCH Issued'] = int(values[-3])    
                    metrics['L2_PREFETCH Useful'] = int(values[-2]) 
                    metrics['L2_PREFETCH Useless'] = int(values[-1])    

    return metrics

In [13]:
prefetchers = ['ip-stride', 'next-line', 'no', 'va-ampm-lite', 'adapt-dist']
bmarks = ['bfs', 'dfs', 'spmv']

benchmark_results = [
    (f'output/{prefetcher}/{bmark}_{prefetcher}.txt', bmark, prefetcher)
    for prefetcher in prefetchers
    for bmark in bmarks
]

metrics_list = [parse_champsim_output(file, benchmark, prefetcher) for file, benchmark, prefetcher in benchmark_results]

df = pd.DataFrame(metrics_list)

df_long = df.melt(id_vars=["Benchmark", "Prefetcher"], var_name="Metric", value_name="Value")

In [14]:
import plotly.graph_objects as go 

textures1 = {
    "L2_PREFETCH Useful": ".",  
    "L2_PREFETCH Useless": "x"     
}

pattern = r"L2_PREFETCH (Useful|Useless)"
df_prefetch = df_long[df_long["Metric"].str.contains(pattern, na=False)].copy()
df_prefetch = df_prefetch[df_prefetch["Prefetcher"] != "no"]

df_prefetch['Metric_Type'] = df_prefetch["Metric"].str.extract(r"(Useful|Useless)")
prefetcher_colors={
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
    # "no": "rgb(144, 238, 144)"
}
data = []

for prefetcher in df_prefetch["Prefetcher"].unique():
    for metric_type in df_prefetch['Metric_Type'].unique():
        metric_data = df_prefetch[(df_prefetch["Prefetcher"] == prefetcher) & 
                                  (df_prefetch["Metric_Type"] == metric_type)]
        
        if metric_data.empty:
            continue
        
        x_data = metric_data["Benchmark"]
        y_data = metric_data["Value"]
        
        data.append(
            go.Bar(
                x=x_data,
                y=y_data,
                hovertext=f"{prefetcher}<br>{metric_type}",
                marker=dict(
                    color=prefetcher_colors.get(prefetcher, "gray"), 
                    pattern_shape=textures1.get(f"L2_PREFETCH {metric_type}", "")  
                ),
                offsetgroup=prefetcher,
                legendgroup=prefetcher,
                showlegend=False
            )
        )

for prefetcher, color in prefetcher_colors.items():
    data.append(
        go.Bar(
            x=[None], y=[None],  
            name=f"{prefetcher}", 
            marker=dict(
                color=color,  
                pattern_shape=None  
            ),
            showlegend=True, 
            legendgroup="Prefetcher",  
        )
    )

for metric_type, texture in textures1.items():
    metric_type_clean = metric_type.replace("L2_PREFETCH ", "")
    data.append(
        go.Bar(
            x=[None], y=[None],
            name=f"{metric_type_clean}",  
            marker=dict(
                pattern_shape=texture, 
                color='rgba(255,255,255,0)' 
            ),
            showlegend=True,
            legendgroup=metric_type, 
        )
    )

layout = go.Layout(
    title='L2C Useful/Useless Prefetch Requests by Prefetcher',
    xaxis=dict(title='Benchmark'),
    yaxis=dict(title='Requests'),
    barmode='group', 
    legend=dict(title="Prefetcher")
)

fig = go.Figure(data=data, layout=layout)
fig.show()

/var/folders/cf/4gvf1xvn2d5d2txbc84b_0880000gn/T/ipykernel_4367/56810239.py:9: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



In [15]:
import plotly.graph_objects as go 

textures1 = {
    "L2_PREFETCH Hit": ".",  
    "L2_PREFETCH Miss": "x"     
}

pattern = r"L2_PREFETCH (Hit|Miss)"
df_prefetch = df_long[df_long["Metric"].str.contains(pattern, na=False)].copy()
df_prefetch = df_prefetch[df_prefetch["Prefetcher"] != "no"]

df_prefetch['Metric_Type'] = df_prefetch["Metric"].str.extract(r"(Hit|Miss)")
prefetcher_colors={
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
    # "no": "rgb(144, 238, 144)"
}
data = []

for prefetcher in df_prefetch["Prefetcher"].unique():
    for metric_type in df_prefetch['Metric_Type'].unique():
        metric_data = df_prefetch[(df_prefetch["Prefetcher"] == prefetcher) & 
                                  (df_prefetch["Metric_Type"] == metric_type)]
        
        if metric_data.empty:
            continue
        
        x_data = metric_data["Benchmark"]
        y_data = metric_data["Value"]
        
        data.append(
            go.Bar(
                x=x_data,
                y=y_data,
                hovertext=f"{prefetcher}<br>{metric_type}",
                marker=dict(
                    color=prefetcher_colors.get(prefetcher, "gray"), 
                    pattern_shape=textures1.get(f"L2_PREFETCH {metric_type}", "")  
                ),
                offsetgroup=prefetcher,
                legendgroup=prefetcher,
                showlegend=False
            )
        )

for prefetcher, color in prefetcher_colors.items():
    data.append(
        go.Bar(
            x=[None], y=[None],  
            name=f"{prefetcher}", 
            marker=dict(
                color=color,  
                pattern_shape=None  
            ),
            showlegend=True, 
            legendgroup="Prefetcher",  
        )
    )

for metric_type, texture in textures1.items():
    metric_type_clean = metric_type.replace("L2_PREFETCH ", "")
    data.append(
        go.Bar(
            x=[None], y=[None],
            name=f"{metric_type_clean}",  
            marker=dict(
                pattern_shape=texture, 
                color='rgba(255,255,255,0)' 
            ),
            showlegend=True,
            legendgroup=metric_type, 
        )
    )

layout = go.Layout(
    title='L2C Prefetch Hit/Miss',
    xaxis=dict(title='Benchmark'),
    yaxis=dict(title='Requests'),
    barmode='group', 
    legend=dict(title="Prefetcher")
)

fig = go.Figure(data=data, layout=layout)
fig.show()

/var/folders/cf/4gvf1xvn2d5d2txbc84b_0880000gn/T/ipykernel_4367/914220797.py:9: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.

